# SmartVision AI — Phase 3: YOLOv8 object detection (25 classes)

**Colab without switching runtimes:** use [`retrain_all_colab.ipynb`](retrain_all_colab.ipynb) (CNNs + this phase + comparison).

Fine-tune **YOLOv8m** (COCO-pretrained) on the same 25-class subset as run 1.

**Runtime:** Colab **T4 GPU**.

**Reuse Drive (do not rebuild the dataset):**
- Unzip `MyDrive/smartvision_dataset.zip` (same zip).
- Run-1 `yolov8_best.pt` stays on Drive. This notebook writes a **new** run folder `yolo_runs/smartvision_yolov8m_freeze/` and only replaces `models/yolov8_best.pt` after the new val mAP is computed. Run-1 weights are copied to `yolov8_best_run1.pt` first.

**Why this recipe:** run 1 re-inited a 25-class head and trained the whole net at `lr0=0.001`, which wiped COCO features (val mAP@0.5 = 56%). This pass **freezes the backbone** (`freeze=10`), uses **YOLOv8m**, and a **lower LR**.

Rubric floor: **mAP@0.5 > 75%** on **val**.

If still ≤ 75% after the freeze run, the last cell does a short unfrozen pass at `lr0=1e-4`.


In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip -q install ultralytics pyyaml pandas matplotlib seaborn pillow opencv-python-headless

In [ ]:
import os, sys, json, time, random, shutil, zipfile
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/Smart_Vision_AI")
    PROJECT_ROOT.mkdir(exist_ok=True)
    zip_path = Path("/content/drive/MyDrive/smartvision_dataset.zip")
    data_dir = PROJECT_ROOT / "smartvision_dataset"
    OUT_ROOT = Path("/content/drive/MyDrive/SmartVision_artifacts")
else:
    PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
    zip_path = PROJECT_ROOT / "smartvision_dataset.zip"
    data_dir = PROJECT_ROOT / "smartvision_dataset"
    OUT_ROOT = PROJECT_ROOT

print("zip exists:", zip_path.exists(), zip_path)
if zip_path.exists():
    print("zip size MB:", round(zip_path.stat().st_size / 1e6, 1))


def unzip_smartvision(zip_path, project_root, data_dir):
    """Notebook 1 zips the *contents* of smartvision_dataset, so the archive
    has classification/ and detection/ at the root — extract into data_dir."""
    with zipfile.ZipFile(zip_path) as z:
        names = [n.replace("\\", "/") for n in z.namelist()]
        print("zip sample:", names[:12])
        nested = any(n.startswith("smartvision_dataset/") for n in names)
        dest = project_root if nested else data_dir
        dest.mkdir(parents=True, exist_ok=True)
        print("Extracting to", dest)
        z.extractall(dest)


def find_detection_dir(project_root):
    candidates = [
        project_root / "smartvision_dataset" / "detection",
        project_root / "detection",
        Path("/content/smartvision_dataset/detection"),
        Path("/content/detection"),
        Path("/content/drive/MyDrive/smartvision_dataset/detection"),
        Path("/content/drive/MyDrive/SmartVision_artifacts/smartvision_dataset/detection"),
    ]
    print("Looking for data.yaml in:")
    for c in candidates:
        ok = (c / "data.yaml").exists()
        print(" ", "FOUND" if ok else "miss ", c)
        if ok:
            return c
    for root in (project_root, Path("/content"), Path("/content/drive/MyDrive")):
        if not root.exists():
            continue
        hits = list(root.rglob("data.yaml"))[:20]
        print("rglob data.yaml under", root, "->", hits)
        for h in hits:
            if h.parent.name == "detection":
                return h.parent
    return None


if zip_path.exists() and not (data_dir / "detection" / "data.yaml").exists():
    print("Unzipping (reuse Drive zip)...")
    unzip_smartvision(zip_path, PROJECT_ROOT, data_dir)
elif not zip_path.exists():
    print("WARNING: Drive zip not found. Finish Smartvision.ipynb last cell first.")

DET = find_detection_dir(PROJECT_ROOT)
print("DET =", DET)

MODELS_DIR = OUT_ROOT / "models"
FIGURES = OUT_ROOT / "reports" / "figures"
REPORTS = OUT_ROOT / "reports"
for p in (MODELS_DIR, FIGURES, REPORTS):
    p.mkdir(parents=True, exist_ok=True)

old_pt = MODELS_DIR / "yolov8_best.pt"
run1_pt = MODELS_DIR / "yolov8_best_run1.pt"
if old_pt.exists() and not run1_pt.exists():
    shutil.copy2(old_pt, run1_pt)
    print("Backed up run-1 YOLO ->", run1_pt)
print("Existing YOLO files:")
for p in sorted(MODELS_DIR.glob("yolo*")):
    print(" ", p.name, f"{p.stat().st_size/1e6:.1f} MB")

if DET is None or not (DET / "data.yaml").exists():
    raise FileNotFoundError(
        "data.yaml not found. Unzip MyDrive/smartvision_dataset.zip — do not rebuild notebook 1."
    )

yaml_text = (DET / "data.yaml").read_text(encoding="utf-8")
print(yaml_text)

import yaml
cfg = yaml.safe_load(yaml_text)
cfg["path"] = str(DET.resolve())
assert int(cfg.get("nc", 0)) == 25, cfg
names = cfg["names"]
if isinstance(names, dict):
    class_names = [names[i] for i in range(len(names))]
else:
    class_names = list(names)
assert "train" not in class_names, class_names
assert len(class_names) == 25
(DET / "data.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print("Updated data.yaml path ->", cfg["path"])
print("names:", class_names)

RUN_NAME = "smartvision_yolov8m_freeze"
print("New Ultralytics run folder:", OUT_ROOT / "yolo_runs" / RUN_NAME)
print("Old run-1 folder is left untouched: yolo_runs/smartvision_yolov8s/")


In [ ]:
## Verify YOLO labels against images (data-driven sanity, not assumed)

import yaml
from pathlib import Path
from collections import Counter

# Recover DET / class_names if the previous cell was the short unzip snippet
if "DET" not in dir() or DET is None:
    hits = list(Path("/content").rglob("data.yaml"))
    DET = next((h.parent for h in hits if h.parent.name == "detection"), None)
    print("Recovered DET =", DET)
assert DET is not None and (DET / "data.yaml").exists(), DET

if "class_names" not in dir():
    cfg = yaml.safe_load((DET / "data.yaml").read_text(encoding="utf-8"))
    names = cfg["names"]
    class_names = [names[i] for i in range(len(names))] if isinstance(names, dict) else list(names)
    print("Loaded class_names from yaml:", class_names)

print("DET =", DET)
print("n classes =", len(class_names))

def verify_split(split):
    img_dir = DET / "images" / split
    lab_dir = DET / "labels" / split
    images = sorted(img_dir.glob("*.jpg"))
    missing, bad, objs = [], 0, 0
    per_class = Counter()
    for imgp in images:
        lab = lab_dir / (imgp.stem + ".txt")
        if not lab.exists():
            missing.append(imgp.name)
            continue
        for line in lab.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) != 5:
                bad += 1
                continue
            cls, xc, yc, w, h = int(parts[0]), *map(float, parts[1:])
            if not (0 <= cls < 25):
                bad += 1
                continue
            if not (0 <= xc <= 1 and 0 <= yc <= 1 and 0 < w <= 1 and 0 < h <= 1):
                bad += 1
                continue
            objs += 1
            per_class[class_names[cls]] += 1
    return {"n_images": len(images), "n_labels": len(list(lab_dir.glob("*.txt"))),
            "missing_labels": len(missing), "bad_rows": bad, "objects": objs, "per_class": dict(per_class)}

for split in ("train", "val", "test"):
    stats = verify_split(split)
    print(split, {k: stats[k] for k in stats if k != "per_class"})
print("\\nTrain objects per class:")
print(pd.Series(verify_split("train")["per_class"]).reindex(class_names))

In [ ]:
## Train YOLOv8m with frozen backbone (keep COCO features; new 25-class head)

from ultralytics import YOLO

# If T4 OOMs, drop batch from 8 to 4 and re-run this cell only.
BATCH = 8
RUN_NAME = "smartvision_yolov8m_freeze"

model = YOLO("yolov8m.pt")
results = model.train(
    data=str(DET / "data.yaml"),
    epochs=40,
    imgsz=640,
    batch=BATCH,
    freeze=10,
    patience=15,
    seed=42,
    workers=2,
    project=str(OUT_ROOT / "yolo_runs"),
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,
    optimizer="AdamW",
    lr0=0.0003,
    lrf=0.01,
    close_mosaic=10,
    mixup=0.1,
    plots=True,
    verbose=True,
)
print(results)
print("Train finished. Next cell evaluates val mAP@0.5 (need > 0.75).")


In [ ]:
## Evaluate on val (primary / rubric) and test (held-out report)

RUN_NAME = "smartvision_yolov8m_freeze"
best = Path(OUT_ROOT / "yolo_runs" / RUN_NAME / "weights" / "best.pt")
print("best exists", best.exists(), best)
if not best.exists():
    raise FileNotFoundError(f"Missing {best}. Re-run the train cell; check yolo_runs/{RUN_NAME}/weights/")

from ultralytics import YOLO
trained = YOLO(str(best))

val_metrics = trained.val(data=str(DET / "data.yaml"), split="val", plots=True)
print("VAL", val_metrics.results_dict if hasattr(val_metrics, "results_dict") else val_metrics)

test_metrics = trained.val(data=str(DET / "data.yaml"), split="test", plots=True)
print("TEST", test_metrics.results_dict if hasattr(test_metrics, "results_dict") else test_metrics)

def extract(m):
    d = m.results_dict if hasattr(m, "results_dict") else {}
    return {
        "map50": float(d.get("metrics/mAP50(B)", getattr(getattr(m, "box", None), "map50", 0.0) or 0.0)),
        "map50_95": float(d.get("metrics/mAP50-95(B)", getattr(getattr(m, "box", None), "map", 0.0) or 0.0)),
        "precision": float(d.get("metrics/precision(B)", getattr(getattr(m, "box", None), "mp", 0.0) or 0.0)),
        "recall": float(d.get("metrics/recall(B)", getattr(getattr(m, "box", None), "mr", 0.0) or 0.0)),
        "raw": {k: float(v) if isinstance(v, (int, float, np.floating)) else str(v) for k, v in d.items()},
    }

val_ex = extract(val_metrics)
test_ex = extract(test_metrics)
print("VAL extracted", {k: val_ex[k] for k in ("map50", "map50_95", "precision", "recall")})
print("TEST extracted", {k: test_ex[k] for k in ("map50", "map50_95", "precision", "recall")})
print("RUBRIC: val mAP@0.5 > 0.75 ?", val_ex["map50"] > 0.75)

ap_per_class = {}
box = getattr(val_metrics, "box", None)
if box is not None and hasattr(box, "ap_class_index") and hasattr(box, "ap50"):
    for idx, ap in zip(box.ap_class_index, box.ap50):
        ap_per_class[class_names[int(idx)]] = float(ap)
print("per-class AP50", ap_per_class)


In [ ]:
## Speed (FPS) on a handful of val images

import time
val_imgs = sorted((DET / "images" / "val").glob("*.jpg"))[:50]
# warmup
_ = trained.predict(source=str(val_imgs[0]), verbose=False)
t0 = time.perf_counter()
_ = trained.predict(source=[str(p) for p in val_imgs], verbose=False)
dt = time.perf_counter() - t0
fps = len(val_imgs) / max(dt, 1e-6)
print(f"{len(val_imgs)} images in {dt:.2f}s -> {fps:.1f} FPS")

In [ ]:
## Visualize predictions on sample val images

samples = val_imgs[:8]
pred = trained.predict(source=[str(p) for p in samples], conf=0.5, verbose=False)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, r, p in zip(axes.ravel(), pred, samples):
    plotted = r.plot()  # BGR ndarray
    ax.imshow(plotted[:, :, ::-1])
    ax.set_title(p.name, fontsize=8)
    ax.axis("off")
fig.suptitle("YOLOv8 val predictions (conf>=0.5)")
fig.tight_layout()
fig.savefig(FIGURES / "yolo_val_samples.png", dpi=140)
plt.show()

In [ ]:
## Failure gallery: one image at a time (avoids CUDA OOM on T4)

import torch
torch.cuda.empty_cache()

fail, ok = [], []
all_val = sorted((DET / "images" / "val").glob("*.jpg"))
chunk = all_val[:40]
for p in chunk:
    r = trained.predict(source=str(p), conf=0.5, verbose=False)[0]
    n = 0 if r.boxes is None else len(r.boxes)
    confs = [] if n == 0 else [float(c) for c in r.boxes.conf]
    mean_c = float(np.mean(confs)) if confs else 0.0
    rec = {"path": p, "n": n, "mean_conf": mean_c}
    (fail if n == 0 else ok).append(rec)
    del r
torch.cuda.empty_cache()

print(f"scanned {len(chunk)} val images; zero-detection={len(fail)}")
ok.sort(key=lambda d: d["mean_conf"])
gallery = fail[:8] + ok[: max(0, 8 - len(fail[:8]))]
if gallery:
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for ax, rec in zip(axes.ravel(), gallery):
        r = trained.predict(source=str(rec["path"]), conf=0.25, verbose=False)[0]
        ax.imshow(r.plot()[:, :, ::-1])
        ax.set_title(f"n={rec['n']} conf={rec['mean_conf']:.2f}", fontsize=8)
        ax.axis("off")
        del r
    fig.suptitle("Failure / low-confidence cases (drawn at conf=0.25)")
    fig.tight_layout()
    fig.savefig(FIGURES / "yolo_failures.png", dpi=140)
    plt.show()
    torch.cuda.empty_cache()
else:
    print("No failures in the scanned slice.")


In [ ]:
## Copy best weights + write yolo_metrics.json
# If val mAP50 is still <= 0.75, short unfrozen pass at low LR (does not delete the freeze run).

map50 = val_ex["map50"]
print("val mAP50 =", map50)
if map50 <= 0.75:
    print("Below 75% floor — 15 more epochs from best.pt with freeze=0, lr0=1e-4")
    from ultralytics import YOLO
    model2 = YOLO(str(best))
    model2.train(
        data=str(DET / "data.yaml"),
        epochs=15,
        imgsz=640,
        batch=8,
        freeze=0,
        patience=8,
        seed=42,
        workers=2,
        project=str(OUT_ROOT / "yolo_runs"),
        name="smartvision_yolov8m_unfreeze",
        exist_ok=True,
        optimizer="AdamW",
        lr0=0.0001,
        lrf=0.01,
        plots=True,
    )
    extra = Path(OUT_ROOT / "yolo_runs" / "smartvision_yolov8m_unfreeze" / "weights" / "best.pt")
    if extra.exists():
        best = extra
        trained = YOLO(str(best))
        val_metrics = trained.val(data=str(DET / "data.yaml"), split="val", plots=True)
        val_ex = extract(val_metrics)
        test_metrics = trained.val(data=str(DET / "data.yaml"), split="test", plots=True)
        test_ex = extract(test_metrics)
        print("VAL after unfreeze pass", {k: val_ex[k] for k in ("map50", "map50_95", "precision", "recall")})
        print("RUBRIC after unfreeze: val mAP@0.5 > 0.75 ?", val_ex["map50"] > 0.75)

dest = MODELS_DIR / "yolov8_best.pt"
shutil.copy2(best, dest)
print("Copied", dest)
print("Run-1 backup still at", MODELS_DIR / "yolov8_best_run1.pt")

yolo_payload = {
    "model": "YOLOv8m",
    "weights": str(dest),
    "val": {k: v for k, v in val_ex.items() if k != "raw"},
    "test": {k: v for k, v in test_ex.items() if k != "raw"},
    "fps": float(fps),
    "per_class_ap50": ap_per_class,
    "n_classes": 25,
    "class_names": class_names,
    "meets_map50_floor": bool(val_ex["map50"] > 0.75),
}
(REPORTS / "yolo_metrics.json").write_text(json.dumps(yolo_payload, indent=2), encoding="utf-8")
print(json.dumps(yolo_payload, indent=2)[:2000])
print("Paste VAL extracted mAP50 back into chat when done.")
